# 24 — End-to-End NLP Project: Support Ticket Routing

**Learning objective.** Run a complete train/validation/test lifecycle from data audit through model selection, test evaluation, serialization and inference.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**raw labeled text → split/preprocess/train/select → locked pipeline → final test/inference**

Follow the information transformation first; treat the API as an implementation detail.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Fit vectorizer **before split** | test information enters representation | metrics become optimistic |
| Change model using test repeatedly | test becomes validation | final estimate loses meaning |
| Serialize preprocessing + model together | inference uses same learned transformation | train/serve skew decreases |

> Write down what should move downstream before changing a control.

## Think before running the next cell

1. Why must TF-IDF vocabulary be learned only from training data?
2. What exactly becomes invalid if you tune on the test set?

### When to use
Use this lifecycle pattern for any supervised NLP system that must be reproducible and deployable.

### When not to use / caution
Do not cargo-cult a random split when time/entity/group structure requires another validation design.

### Debugging lens
For any suspicious score, audit data lineage and fit boundaries before tuning the model.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


## Problem contract
Route support tickets into `billing`, `access`, `technical`, or `account`. We will preserve a final test set, choose between candidate models using validation macro-F1, then evaluate the chosen pipeline once on test.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from joblib import dump, load

df=pd.read_csv(DATA/'customer_support.csv').drop_duplicates().reset_index(drop=True)
print('rows:',len(df),'duplicates after cleaning:',df.duplicated().sum())
print(df.label.value_counts())
trainval,test=train_test_split(df,test_size=.20,random_state=42,stratify=df.label)
train,val=train_test_split(trainval,test_size=.25,random_state=42,stratify=trainval.label) # 60/20/20
print('split sizes:',len(train),len(val),len(test))

rows: 40 duplicates after cleaning: 0
label
billing      10
access       10
technical    10
account      10
Name: count, dtype: int64
split sizes: 24 8 8


In [3]:
def make(model):
    return Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True)),('model',model)])
candidates={
 'logreg':make(LogisticRegression(max_iter=1000,random_state=42)),
 'linear_svm':make(LinearSVC(random_state=42))}
val_scores={}
for name,m in candidates.items():
    m.fit(train.text,train.label)
    val_scores[name]=f1_score(val.label,m.predict(val.text),average='macro')
print('validation macro-F1:',{k:round(v,3) for k,v in val_scores.items()})
best_name=max(val_scores,key=val_scores.get)
print('selected:',best_name)

validation macro-F1: {'logreg': 0.867, 'linear_svm': 0.867}
selected: logreg


In [4]:
# Refit selected configuration on train+validation only after model selection.
best=make(LogisticRegression(max_iter=1000,random_state=42)) if best_name=='logreg' else make(LinearSVC(random_state=42))
best.fit(trainval.text,trainval.label)
test_pred=best.predict(test.text)
print('FINAL TEST REPORT')
print(classification_report(test.label,test_pred,zero_division=0))
print('confusion matrix labels:',sorted(df.label.unique()))
print(confusion_matrix(test.label,test_pred,labels=sorted(df.label.unique())))

FINAL TEST REPORT
              precision    recall  f1-score   support

      access       1.00      1.00      1.00         2
     account       1.00      1.00      1.00         2
     billing       1.00      1.00      1.00         2
   technical       1.00      1.00      1.00         2

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8

confusion matrix labels: ['access', 'account', 'billing', 'technical']
[[2 0 0 0]
 [0 2 0 0]
 [0 0 2 0]
 [0 0 0 2]]


In [5]:
artifact=Path('support_router.joblib')
dump(best,artifact)
reloaded=load(artifact)
requests=['My card payment appears twice','The app freezes whenever I open it','Please change the email on my account']
print(pd.DataFrame({'request':requests,'route':reloaded.predict(requests)}).to_string(index=False))
artifact.unlink()

                              request     route
        My card payment appears twice   billing
   The app freezes whenever I open it technical
Please change the email on my account   account


### Leakage controls used
- Split before fitting vectorizers/models.
- Candidate choice used validation only.
- Test was touched only after selection.
- The serialized artifact is the whole preprocessing+model Pipeline, preventing train/inference preprocessing divergence.

In [6]:

import matplotlib.pyplot as plt
labels=sorted(df.label.unique()); cm=confusion_matrix(test.label,test_pred,labels=labels)
fig, ax = plt.subplots(figsize=(5,4)); im=ax.imshow(cm)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title('Final test confusion matrix')
for i in range(len(labels)):
    for j in range(len(labels)): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()


[static visualization generated successfully during execution; rerun in Jupyter/VS Code to display]


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Run train/validation/test correctly
- Serialize the entire text pipeline
- Separate model selection from final test reporting